In [9]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
from matplotlib.lines import Line2D
sys.path.insert(0,'..')
from scripts.utils import Config,load_stats
from scripts.models.sr.train import load_features
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [10]:
CONFIG      = Config()
STATS       = load_stats(CONFIG.splitsdir)
SPLITSDIR   = CONFIG.splitsdir
PREDSDIR    = CONFIG.predsdir
NNRUNS      = CONFIG.nn['runs']
EQUATIONS   = CONFIG.sr['optimizedeqs']
ATMRUN      = CONFIG.sr['runs']['sr_atm']
LABELS      = {**{name:run['description'] for name,run in NNRUNS.items()},**{name:eq['description'] for name,eq in EQUATIONS.items()}}
COLORS      = {**{name:run['color'] for name,run in NNRUNS.items()},**{name:eq['color'] for name,eq in EQUATIONS.items()}}
CORRECTIONS = ['none','constant','binned']
STYLES      = {'none':'-','binned':'--'}
FITSPLIT    = 'valid'
SPLIT       = 'test'
NSMEARBINS  = 20
BLBINS      = 40
MINSAMPLES  = 50
MINPRECIP   = 0.1
LANDFRAC    = 0.5
TOPQUANTILE = 0.8
NHOURBINS   = 8
TIMEWINDOW  = CONFIG.timewindow

In [11]:
def load_predictions(name,split):
    path = os.path.join(PREDSDIR,f'{name}_{split}_predictions.nc')
    if not os.path.exists(path):
        return None
    with xr.open_dataset(path) as ds:
        da = ds['tp'].load()
    if 'seed' in da.dims:
        return da.transpose('seed','time','lat','lon').values.reshape(da.sizes['seed'],-1)
    return da.squeeze().transpose('time','lat','lon').values.reshape(1,-1)

def load_truth(split):
    with xr.open_dataset(os.path.join(SPLITSDIR,f'{split}.h5'),engine='h5netcdf') as ds:
        return ds['tp'].transpose('time','lat','lon').values.ravel()

def fit_smearing(pred,true):
    wet    = np.isfinite(pred)&np.isfinite(true)&(pred>0)
    ratio  = np.exp(np.log1p(true[wet])-np.log1p(pred[wet]))
    edges  = np.quantile(pred[wet],np.linspace(0,1,NSMEARBINS+1))
    binidx = np.clip(np.digitize(pred[wet],edges[1:-1]),0,NSMEARBINS-1)
    binned = np.bincount(binidx,weights=ratio,minlength=NSMEARBINS)/np.maximum(np.bincount(binidx,minlength=NSMEARBINS),1)
    return dict(constant=float(ratio.mean()),edges=edges,binned=binned)

def apply_smearing(pred,smear,kind):
    if kind=='none':
        return pred
    out = pred.copy()
    wet = np.isfinite(pred)&(pred>0)
    factor = smear['constant'] if kind=='constant' else smear['binned'][np.clip(np.digitize(pred[wet],smear['edges'][1:-1]),0,NSMEARBINS-1)]
    out[wet] = factor*(1+pred[wet])-1
    return out

def calc_scores(pred,true):
    valid = np.isfinite(pred)&np.isfinite(true)
    p,t   = pred[valid],true[valid]
    lp,lt = np.log1p(p),np.log1p(t)
    return dict(r2=1-np.sum((p-t)**2)/np.sum((t-t.mean())**2),mse=np.mean((p-t)**2),
                logr2=1-np.sum((lp-lt)**2)/np.sum((lt-lt.mean())**2),logmse=np.mean((lp-lt)**2))

def calc_diurnal_cycle(values,mask):
    counts = np.bincount(hourbin[mask],minlength=NHOURBINS)
    return np.bincount(hourbin[mask],weights=values[mask],minlength=NHOURBINS)/np.maximum(counts,1)

In [12]:
truefit,truetest = load_truth(FITSPLIT),load_truth(SPLIT)
corrected,smearing,rows = {},{},[]
for name in LABELS:
    fitpreds,testpreds = load_predictions(name,FITSPLIT),load_predictions(name,SPLIT)
    if fitpreds is None or testpreds is None:
        print(f'Skipping {LABELS[name]}, missing {FITSPLIT} or {SPLIT} predictions')
        continue
    smears = [fit_smearing(fitpred,truefit) for fitpred in fitpreds]
    corrected[name] = {kind:np.stack([apply_smearing(testpred,smear,kind) for testpred,smear in zip(testpreds,smears)]) for kind in CORRECTIONS}
    smearing[name]  = smears
    scores = {kind:pd.DataFrame([calc_scores(pred,truetest) for pred in corrected[name][kind]]).mean() for kind in CORRECTIONS}
    rows.append(dict(model=LABELS[name],factor=np.mean([smear['constant'] for smear in smears]),
                     logr2=scores['none']['logr2'],logmse=scores['none']['logmse'],
                     **{f'r2_{kind}':scores[kind]['r2'] for kind in CORRECTIONS},**{f'mse_{kind}':scores[kind]['mse'] for kind in CORRECTIONS}))
table = pd.DataFrame(rows).set_index('model').sort_values('r2_none')
pd.set_option('display.width',250)
print(table.round(4).to_string())
for name in ['sr_bl_eq','sr_atm_eq','sr_all_pc_eq']:
    if name in smearing:
        print(f"{LABELS[name]} binned factors (lowest to highest predicted precipitation): {np.round(np.mean([smear['binned'] for smear in smearing[name]],axis=0),3)}")

Skipping NN-BL, missing valid or test predictions
Skipping NN-FULL, missing valid or test predictions
Skipping NN-NONPARAM, missing valid or test predictions
Skipping NN-GAUSS, missing valid or test predictions
Skipping SR-BL, missing valid or test predictions
Skipping SR-ATM, missing valid or test predictions
Skipping SR-SFC, missing valid or test predictions
Skipping SR-ALL, missing valid or test predictions
Skipping SR-ALL-PC, missing valid or test predictions


KeyError: "None of ['model'] are in the columns"

In [13]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = (ds.sizes[dim] for dim in ['time','lat','lon'])
    lon   = ds['lon'].values
    hours = ds['time'].dt.hour.values
    bl,shf = (ds[name].transpose('time','lat','lon').values.ravel() for name in ['bl','shf'])
    lf    = xr.broadcast(ds['lf'],ds['tp'])[0].transpose('time','lat','lon').values.ravel().astype(np.float64)
thetae = load_features(SPLIT,ATMRUN,CONFIG)[0]['thetae'].values
localhour = (hours[:,None,None]+TIMEWINDOW/2+lon[None,None,:]/15.0)%24
hourbin   = (np.broadcast_to(localhour,(ntime,nlat,nlon)).ravel()//(24/NHOURBINS)).astype(int)
hourcenters = (np.arange(NHOURBINS)+0.5)*24/NHOURBINS
seedmean  = {name:{kind:np.nanmean(preds,axis=0) for kind,preds in kinds.items()} for name,kinds in corrected.items()}

finite   = np.isfinite(bl)&np.isfinite(truetest)
edges    = np.linspace(*np.percentile(bl[finite],[1,99]),BLBINS+1)
centers  = 0.5*(edges[:-1]+edges[1:])
binidx   = np.clip(np.digitize(bl[finite],edges)-1,0,BLBINS-1)
counts   = np.bincount(binidx,minlength=BLBINS)
binmean  = lambda values:np.where(counts>=MINSAMPLES,np.bincount(binidx,weights=values[finite],minlength=BLBINS)/np.maximum(counts,1),np.nan)
blcurves = {'ERA5':binmean(truetest),**{kind:binmean(seedmean['sr_bl_eq'][kind]) for kind in CORRECTIONS}}
top      = np.where(np.isfinite(blcurves['ERA5']))[0][-5:]
print('SR-BL bin means in the five highest populated B_L bins (mm):')
print(pd.DataFrame({'B_L':centers[top],**{key:curve[top] for key,curve in blcurves.items()}}).round(3).to_string(index=False))

cyclemodels = ['sr_atm_eq','sr_sfc_eq','sr_all_pc_eq']
regions     = {'Ocean':lf<LANDFRAC,'Land':lf>=LANDFRAC}
cycles      = {}
rows        = []
for region,regionmask in regions.items():
    mask = regionmask&np.isfinite(truetest)&np.all([np.isfinite(seedmean[name]['none']) for name in cyclemodels],axis=0)
    cycles[(region,'ERA5')] = calc_diurnal_cycle(truetest,mask)
    era5amp = np.ptp(cycles[(region,'ERA5')])
    for name in cyclemodels:
        for kind in CORRECTIONS:
            cycles[(region,name,kind)] = calc_diurnal_cycle(seedmean[name][kind],mask)
            rows.append(dict(region=region,model=LABELS[name],correction=kind,
                             meanratio=cycles[(region,name,kind)].mean()/cycles[(region,'ERA5')].mean(),
                             amplitudepct=100*np.ptp(cycles[(region,name,kind)])/era5amp,
                             peakhour=hourcenters[np.argmax(cycles[(region,name,kind)])]))
    rows.append(dict(region=region,model='ERA5',correction='none',meanratio=1.0,amplitudepct=100.0,peakhour=hourcenters[np.argmax(cycles[(region,'ERA5')])]))
print('Diurnal cycle relative to ERA5 (mean ratio, amplitude as % of ERA5, peak LST):')
print(pd.DataFrame(rows).round(3).to_string(index=False))

oceanmask = (lf<LANDFRAC)&np.isfinite(truetest)&np.isfinite(thetae)&np.isfinite(shf)&(seedmean['sr_atm_eq']['none']>=MINPRECIP)
strong    = oceanmask&(shf>=np.quantile(shf[oceanmask],TOPQUANTILE))&(thetae>=np.quantile(thetae[oceanmask],TOPQUANTILE))
print(f'Ocean samples in the top {100*(1-TOPQUANTILE):.0f}% of both SHF and kernel-integrated thetae (SR-ATM >= {MINPRECIP} mm): {strong.sum():,}')
print(f'   ERA5 mean: {truetest[strong].mean():.3f} mm')
for name in ['sr_atm_eq','sr_all_pc_eq']:
    print(f"   {LABELS[name]}: "+', '.join(f'{kind} {seedmean[name][kind][strong].mean():.3f}' for kind in CORRECTIONS)+' mm')

KeyError: 'sr_bl_eq'

In [ ]:
handles = [Line2D([],[],color='gray',linewidth=2,label='ERA5')]+[Line2D([],[],color=COLORS[name],linewidth=2,label=LABELS[name]) for name in ['sr_bl_eq','sr_atm_eq','sr_all_pc_eq']]
handles+= [Line2D([],[],color='k',linestyle=STYLES['none'],linewidth=2,label='Uncorrected'),Line2D([],[],color='k',linestyle=STYLES['binned'],linewidth=2,label='Binned smearing')]
fig,axs = pplt.subplots(ncols=3,refwidth=2.2,share=False,wspace=1.5)
axs[0].scatter(centers,blcurves['ERA5'],color='gray',markersize=12,zorder=4)
for kind,style in STYLES.items():
    axs[0].plot(centers,blcurves[kind],color=COLORS['sr_bl_eq'],linestyle=style,linewidth=2)
axs[0].format(title='Precipitation vs $\\mathit{B_L}$',xlabel='$\\mathit{B_L}$ (m/s$^2$)',ylabel='Total Precipitation (mm)')
for ax,region in zip(axs[1:],regions):
    ax.plot(hourcenters,cycles[(region,'ERA5')],color='gray',linewidth=2)
    for name in ['sr_atm_eq','sr_all_pc_eq']:
        for kind,style in STYLES.items():
            ax.plot(hourcenters,cycles[(region,name,kind)],color=COLORS[name],linestyle=style,linewidth=2)
    ax.format(title=f'Diurnal Cycle ({region})',xlim=(0,24),xticks=[6,12,18],xticklabels=['06 LST','12 LST','18 LST'],ylim=(0,2),ylabel='Total Precipitation (mm)')
fig.legend(handles,loc='b',ncols=3)
fig.format(abc=True,titleloc='l',grid=False)
pplt.show()